### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
## Open AI API Key and Open Source models -- LLama3, Gemma2, # Mistral, etc.

import os
from dotenv import load_dotenv

load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

model = ChatGroq(model="gemma2-9b-it", api_key=GROQ_API_KEY)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7c64969e0350>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c64969e16d0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English to Spanish"),
    HumanMessage(content="Hello, how are you?")
]

result = model.invoke(messages)

In [8]:
result

AIMessage(content='Hola, ¿cómo estás? \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 22, 'total_tokens': 32, 'completion_time': 0.018181818, 'prompt_time': 0.001328679, 'queue_time': 0.02076139, 'total_time': 0.019510497}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--861721d7-641d-4d6e-b4ee-4e1ebdedd295-0', usage_metadata={'input_tokens': 22, 'output_tokens': 10, 'total_tokens': 32})

In [9]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(result)

'Hola, ¿cómo estás? \n'

In [11]:
## Using LCEL - chain components

chain = model | parser
chain.invoke(messages)

'Hola, ¿cómo estás? \n'

In [12]:
## Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}"

prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template), ("user", "{text}")]
)

In [16]:
res = prompt.invoke({"language": "Spanish", "text": "Hello"})

In [17]:
res.to_messages()

[SystemMessage(content='Translate the following into Spanish', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [18]:
chain = prompt | model | parser
chain.invoke({"language": "Spanish", "text": "How are you?"})

'The most common way to say "How are you?" in Spanish is:\n\n**¿Cómo estás?** \n\n(pronounced: koh-moh eh-stahs?)\n\nThis is used for informal situations with friends, family, or people you know well. \n\nIf you want to be more formal, you can use:\n\n**¿Cómo está?**\n\n(pronounced: koh-moh eh-stah?)\n\nThis is used for formal situations with strangers, people in authority, or older people.\n\n\nLet me know if you have any other phrases you\'d like me to translate!\n'